In [1]:
import re
import pickle
import pandas as pd
from sentence_transformers import SentenceTransformer

/home/indra/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
movies_path = "../data/ml-1m/movies.dat"
output_path = "../data/ml-1m/content_embeddings.pkl"

df = pd.read_csv(
    movies_path,
    sep="::",
    engine="python",
    header=None,
    encoding="latin-1",
    names=["movie_id", "title", "genres"]
)
print(f"{len(df)} movies loaded")
df.head()

3883 movies loaded


,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
def build_text(row):
    title = row["title"]
    year_match = re.search(r"\((\d{4})\)", title)
    year = year_match.group(1) if year_match else ""
    name = re.sub(r"\(\d{4}\)", "", title).strip()
    genres = row["genres"].replace("|", " ")
    return f"{name} {year} {genres}".strip()

df["text"] = df.apply(build_text, axis=1)
df[["movie_id", "text"]].head(10)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")

In [ ]:
embeddings = model.encode(
    df["text"].tolist(),
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
data = {
    "item_id": df["movie_id"].tolist(),
    "embedding": embeddings.tolist()
}

with open(output_path, "wb") as f:
    pickle.dump(data, f)

print(f"Saved {len(data['item_id'])} embeddings to {output_path}")
print(f"Embedding dim: {len(data['embedding'][0])}")